# ECSC Developmental Analysis v4: Burstiness / Event-Like Long-Range Memory

**New in v4:** Time-resolved analysis to detect "bursty" retrieval patterns

**Core idea:** Instead of one target at the end, compute memory curves at K positions throughout each document to detect whether long-range influence is:
- **Diffuse**: spread smoothly across the narrative
- **Event-like**: concentrated in spikes at specific moments (entity reintroduction, callbacks)

**Metrics:**
- `I_512(t)` = benefit from long context (512) beyond local (32) at each target position
- `spike_mass` = concentration of influence in top 10% of targets
- `spike_count` = number of targets exceeding 95th percentile
- `CV` = coefficient of variation (std/mean) of influence trace

**Keeps:** All v3 end-of-doc analysis unchanged for continuity

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes scipy

In [ ]:
import torch
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy import stats
from tqdm.auto import tqdm
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

class Config:
    # === Data ===
    MIN_WORDS = 200
    
    # === End-of-doc analysis (unchanged from v3) ===
    MIN_TARGET_SIZE = 20
    TARGET_FRACTION = 0.10
    MAX_CONTEXT_FIXED = 512
    MAX_CONTEXT_LONG = 1024
    BENEFIT_EPS = 1.0
    
    # === Burstiness analysis (NEW) ===
    K_TARGETS = 10                    # Number of target positions per doc
    TARGET_START_FRAC = 0.15          # Avoid initial warmup
    TARGET_END_FRAC = 0.90            # Avoid very end (analyzed separately)
    TARGET_REGION_TOKENS = 40         # Fixed target size for burstiness
    
    # Context grid for burstiness (cheap version)
    CONTEXT_GRID_BURST = [32, 128, 256, 384, 512]
    
    # Context grid for half-life trace (richer, optional)
    CONTEXT_GRID_HALFLIFE = [4, 8, 12, 16, 20, 24, 28, 32, 64, 128, 256, 512]
    COMPUTE_HALFLIFE_TRACE = False    # Set True for richer analysis
    
    # Long-range influence computation
    SHORT_CONTEXT = 32                # Baseline for I_L computation
    LONG_CONTEXT = 512                # Long-range context
    
    # Spike detection
    SPIKE_PERCENTILE = 95
    SPIKE_MASS_TOP_FRAC = 0.10        # Top 10% of targets
    
    # Filtering
    MIN_TARGETS_PER_DOC = 6           # Skip docs with too few valid targets
    REQUIRE_CTX_FOR_L = True          # Only compute target if enough context available
    
    # Bootstrap
    N_BOOTSTRAP = 1000
    CI_LEVEL = 0.95
    
    # Plotting
    MIN_DOCS_FOR_PLOT = 10

config = Config()
print(f"Burstiness analysis: K={config.K_TARGETS} targets per doc")
print(f"Target region: {config.TARGET_REGION_TOKENS} tokens (fixed)")
print(f"Context grid: {config.CONTEXT_GRID_BURST}")
print(f"I_L = ppl({config.SHORT_CONTEXT}) - ppl({config.LONG_CONTEXT})")

In [ ]:
from google.colab import files

print("Upload transcripts.jsonl:")
uploaded = files.upload()
DATA_FILE = list(uploaded.keys())[0]
print(f"Uploaded: {DATA_FILE}")

In [ ]:
# Load data
records = []
with open(DATA_FILE) as f:
    for line in f:
        record = json.loads(line)
        record['pop'] = json.loads(record['population'])
        record['word_count'] = len(record['text'].split())
        record['age_months'] = record['pop']['age_months']
        records.append(record)

df_all = pd.DataFrame(records)
df = df_all[df_all['word_count'] >= config.MIN_WORDS].copy()

def age_bin(age):
    if age < 72: return '4-6yr'
    elif age < 96: return '6-8yr'
    elif age < 120: return '8-10yr'
    else: return '10+yr'

df['age_group'] = df['age_months'].apply(age_bin)

print(f"Documents: {len(df)} (of {len(df_all)} total)")
print(f"\nBy age group:")
for ag in ['4-6yr', '6-8yr', '8-10yr', '10+yr']:
    n = len(df[df['age_group'] == ag])
    if n > 0:
        print(f"  {ag}: n={n}")

In [ ]:
MODEL_NAME = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
model.eval()
print(f"Loaded {MODEL_NAME}")

In [ ]:
# ============================================================
# CORE FUNCTIONS (unchanged from v3)
# ============================================================

@torch.no_grad()
def compute_perplexity_on_region(token_ids, target_start, target_end):
    """Compute perplexity on tokens in [target_start, target_end]."""
    if target_end > len(token_ids):
        target_end = len(token_ids)
    
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    logits = outputs.logits[0]
    
    total_loss = 0.0
    count = 0
    
    for i in range(target_start, target_end - 1):
        log_probs = torch.log_softmax(logits[i], dim=-1)
        target_token = token_ids[i + 1]
        token_loss = -log_probs[target_token].item()
        total_loss += token_loss
        count += 1
    
    if count == 0:
        return float('inf'), 0
    
    return np.exp(total_loss / count), total_loss / count


def compute_cumulative_min(contexts, perplexities):
    """Convert to monotone non-increasing curve using cumulative minimum."""
    order = np.argsort(contexts)
    contexts = np.array(contexts)[order]
    perplexities = np.array(perplexities)[order]
    ppl_mon = np.minimum.accumulate(perplexities)
    return contexts, ppl_mon


def interpolate_perplexity(contexts, perplexities, target_ctx):
    """Linearly interpolate perplexity at a target context length."""
    if target_ctx <= contexts[0]:
        return perplexities[0]
    if target_ctx >= contexts[-1]:
        return perplexities[-1]
    
    for i in range(len(contexts) - 1):
        if contexts[i] <= target_ctx <= contexts[i+1]:
            frac = (target_ctx - contexts[i]) / (contexts[i+1] - contexts[i])
            return perplexities[i] + frac * (perplexities[i+1] - perplexities[i])
    
    return perplexities[-1]

In [ ]:
# ============================================================
# BURSTINESS: TARGET SELECTION
# ============================================================

def select_target_positions(n_tokens, config):
    """
    Select K evenly-spaced target positions within [lo, hi] range.
    
    Returns list of target start positions (token indices).
    Each target region will be [pos : pos + TARGET_REGION_TOKENS].
    """
    # Eligible range
    lo = int(config.TARGET_START_FRAC * n_tokens)
    hi = int(config.TARGET_END_FRAC * n_tokens) - config.TARGET_REGION_TOKENS
    
    # Need enough room for K targets
    if hi <= lo:
        return []
    
    # Evenly space K targets
    if config.K_TARGETS == 1:
        positions = [(lo + hi) // 2]
    else:
        step = (hi - lo) / (config.K_TARGETS - 1)
        positions = [int(lo + i * step) for i in range(config.K_TARGETS)]
    
    # Filter: each target needs enough context for LONG_CONTEXT
    if config.REQUIRE_CTX_FOR_L:
        positions = [p for p in positions if p >= config.LONG_CONTEXT]
    
    return positions


# Test target selection
test_positions = select_target_positions(800, config)
print(f"Test doc (800 tokens): {len(test_positions)} targets at positions {test_positions}")

In [ ]:
# ============================================================
# BURSTINESS: LONG-RANGE INFLUENCE TRACE (I_L)
# ============================================================

def compute_influence_at_target(full_tokens, target_pos, context_grid, target_size):
    """
    Compute perplexity at a single target position for multiple context lengths.
    
    Returns dict: {context_length: perplexity}
    """
    results = {}
    
    target_end = target_pos + target_size
    if target_end > len(full_tokens):
        return results
    
    for ctx_len in context_grid:
        # Check if enough context available
        if ctx_len > target_pos:
            continue
        
        # Extract context + target
        start_pos = target_pos - ctx_len
        tokens_to_score = full_tokens[start_pos:target_end]
        
        # Target region within this subsequence
        target_start_local = ctx_len
        target_end_local = ctx_len + target_size
        
        ppl, _ = compute_perplexity_on_region(tokens_to_score, target_start_local, target_end_local)
        results[ctx_len] = ppl
    
    return results


def compute_I_L(ppl_dict, short_ctx, long_ctx):
    """
    Compute long-range influence: I_L = ppl(short) - ppl(long)
    
    Higher I_L means more benefit from long context at this target.
    """
    if short_ctx not in ppl_dict or long_ctx not in ppl_dict:
        return np.nan
    
    return ppl_dict[short_ctx] - ppl_dict[long_ctx]


def compute_burstiness_trace(full_tokens, config):
    """
    Compute long-range influence trace for a document.
    
    Returns:
        positions: list of target positions
        I_L_trace: list of I_L values at each position
        ppl_traces: dict of {context_len: [ppl at each target]}
    """
    n_tokens = len(full_tokens)
    positions = select_target_positions(n_tokens, config)
    
    if len(positions) < config.MIN_TARGETS_PER_DOC:
        return [], [], {}
    
    I_L_trace = []
    ppl_traces = {ctx: [] for ctx in config.CONTEXT_GRID_BURST}
    valid_positions = []
    
    for pos in positions:
        ppl_dict = compute_influence_at_target(
            full_tokens, pos, 
            config.CONTEXT_GRID_BURST, 
            config.TARGET_REGION_TOKENS
        )
        
        # Need both short and long context for I_L
        if config.SHORT_CONTEXT in ppl_dict and config.LONG_CONTEXT in ppl_dict:
            I_L = compute_I_L(ppl_dict, config.SHORT_CONTEXT, config.LONG_CONTEXT)
            I_L_trace.append(I_L)
            valid_positions.append(pos)
            
            for ctx in config.CONTEXT_GRID_BURST:
                ppl_traces[ctx].append(ppl_dict.get(ctx, np.nan))
    
    return valid_positions, I_L_trace, ppl_traces

In [ ]:
# ============================================================
# BURSTINESS: METRICS
# ============================================================

def compute_burstiness_metrics(I_L_trace, config):
    """
    Compute burstiness metrics from an influence trace.
    
    Returns dict with:
        - spike_mass: concentration in top SPIKE_MASS_TOP_FRAC
        - spike_count: number of values above SPIKE_PERCENTILE
        - cv: coefficient of variation
        - kurtosis: excess kurtosis
        - mean, max, std of I_L
    """
    trace = np.array(I_L_trace)
    trace = trace[~np.isnan(trace)]
    
    if len(trace) < 3:
        return {
            'spike_mass': np.nan,
            'spike_count': np.nan,
            'cv': np.nan,
            'kurtosis': np.nan,
            'I_L_mean': np.nan,
            'I_L_max': np.nan,
            'I_L_std': np.nan,
            'n_targets': len(trace),
        }
    
    # Basic stats
    mean_val = np.mean(trace)
    max_val = np.max(trace)
    std_val = np.std(trace)
    
    # Coefficient of variation (handle near-zero mean)
    if abs(mean_val) > 0.01:
        cv = std_val / abs(mean_val)
    else:
        cv = np.nan
    
    # Spike mass concentration
    # Only consider positive values (actual benefit from long context)
    positive_trace = trace[trace > 0]
    if len(positive_trace) > 0:
        total_positive = np.sum(positive_trace)
        n_top = max(1, int(np.ceil(len(trace) * config.SPIKE_MASS_TOP_FRAC)))
        sorted_trace = np.sort(trace)[::-1]  # Descending
        top_sum = np.sum(sorted_trace[:n_top])
        spike_mass = top_sum / total_positive if total_positive > 0 else 0
    else:
        spike_mass = 0
    
    # Spike count (above percentile threshold)
    threshold = np.percentile(trace, config.SPIKE_PERCENTILE)
    spike_count = np.sum(trace > threshold)
    
    # Kurtosis (excess)
    if std_val > 0:
        kurtosis = stats.kurtosis(trace, fisher=True)
    else:
        kurtosis = np.nan
    
    return {
        'spike_mass': spike_mass,
        'spike_count': spike_count,
        'cv': cv,
        'kurtosis': kurtosis,
        'I_L_mean': mean_val,
        'I_L_max': max_val,
        'I_L_std': std_val,
        'n_targets': len(trace),
    }

In [ ]:
# ============================================================
# END-OF-DOC ANALYSIS (unchanged from v3)
# ============================================================

def get_context_lengths(max_context):
    """Generate context lengths with dense sampling at short range."""
    lengths = []
    lengths.extend(range(4, min(33, max_context + 1), 4))
    lengths.extend(range(48, min(129, max_context + 1), 16))
    lengths.extend(range(160, max_context + 1, 32))
    return sorted(set(lengths))


def compute_half_life_robust(contexts, perplexities, max_ctx_fixed=None, benefit_eps=1.0):
    """Compute half-life with cumulative-min envelope and fixed max context."""
    contexts, ppl_mon = compute_cumulative_min(contexts, perplexities)
    
    ppl_at_min = ppl_mon[0]
    ctx_min = contexts[0]
    
    max_ctx_available = contexts[-1]
    if max_ctx_fixed is not None:
        max_ctx_used = min(max_ctx_available, max_ctx_fixed)
    else:
        max_ctx_used = max_ctx_available
    
    ppl_at_max = interpolate_perplexity(contexts, ppl_mon, max_ctx_used)
    total_benefit = ppl_at_min - ppl_at_max
    benefit_ok = total_benefit >= benefit_eps
    
    half_life = np.nan
    if benefit_ok:
        target_ppl = ppl_at_min - 0.5 * total_benefit
        for i in range(len(ppl_mon) - 1):
            if contexts[i+1] > max_ctx_used:
                break
            if ppl_mon[i] >= target_ppl >= ppl_mon[i+1]:
                frac = (ppl_mon[i] - target_ppl) / (ppl_mon[i] - ppl_mon[i+1])
                half_life = contexts[i] + frac * (contexts[i+1] - contexts[i])
                break
    
    ppl_at_32 = interpolate_perplexity(contexts, ppl_mon, 32) if 32 <= max_ctx_available else np.nan
    early_drop = ppl_at_min - ppl_at_32 if not np.isnan(ppl_at_32) else np.nan
    early_drop_pct = 100 * early_drop / total_benefit if (benefit_ok and not np.isnan(early_drop)) else np.nan
    
    early_mask = (contexts >= ctx_min) & (contexts <= 32)
    if early_mask.sum() >= 3:
        slope, _, _, _, _ = stats.linregress(contexts[early_mask], ppl_mon[early_mask])
        early_slope = slope
    else:
        early_slope = np.nan
    
    return {
        'ppl_at_min_ctx': ppl_at_min,
        'ppl_at_32': ppl_at_32,
        'ppl_at_max_ctx': ppl_at_max,
        'total_benefit': total_benefit,
        'benefit_ok': benefit_ok,
        'half_life': half_life,
        'early_drop': early_drop,
        'early_drop_pct': early_drop_pct,
        'early_slope': early_slope,
        'max_ctx_available': max_ctx_available,
        'max_ctx_used': max_ctx_used,
    }


def analyze_document_endofdoc(text):
    """Run end-of-doc context ablation (unchanged from v3)."""
    full_tokens = tokenizer.encode(text)
    n_tokens = len(full_tokens)
    
    target_size = max(config.MIN_TARGET_SIZE, int(n_tokens * config.TARGET_FRACTION))
    target_size = min(target_size, n_tokens - 8)
    
    if target_size < config.MIN_TARGET_SIZE:
        return [], {}
    
    max_context = n_tokens - target_size
    context_lengths = get_context_lengths(max_context)
    
    if len(context_lengths) < 3:
        return [], {}
    
    results = []
    for ctx_len in context_lengths:
        doc_start = n_tokens - target_size - ctx_len
        truncated_tokens = full_tokens[doc_start:]
        target_start = len(truncated_tokens) - target_size
        target_end = len(truncated_tokens)
        ppl, loss = compute_perplexity_on_region(truncated_tokens, target_start, target_end)
        results.append({'context_length': ctx_len, 'perplexity': ppl, 'loss': loss})
    
    meta = {'n_tokens': n_tokens, 'target_size': target_size, 'max_context_available': max_context}
    return results, meta

In [ ]:
# ============================================================
# RUN COMBINED ANALYSIS
# ============================================================

all_results_endofdoc = []  # For end-of-doc curves
doc_metrics = []           # Per-doc metrics (end-of-doc + burstiness)
burst_traces = []          # Per-doc influence traces

age_groups = ['4-6yr', '6-8yr', '8-10yr', '10+yr']

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    full_tokens = tokenizer.encode(row['text'])
    n_tokens = len(full_tokens)
    
    # --- End-of-doc analysis (v3) ---
    endofdoc_results, endofdoc_meta = analyze_document_endofdoc(row['text'])
    
    if not endofdoc_results or len(endofdoc_results) < 3:
        continue
    
    # Store raw end-of-doc results
    for r in endofdoc_results:
        r['doc_id'] = row['doc_id']
        r['age_months'] = row['age_months']
        r['age_group'] = row['age_group']
        all_results_endofdoc.append(r)
    
    # Compute end-of-doc metrics
    contexts = [r['context_length'] for r in endofdoc_results]
    perplexities = [r['perplexity'] for r in endofdoc_results]
    
    metrics_fixed = compute_half_life_robust(
        contexts, perplexities, 
        max_ctx_fixed=config.MAX_CONTEXT_FIXED,
        benefit_eps=config.BENEFIT_EPS
    )
    
    # --- Burstiness analysis (NEW) ---
    positions, I_L_trace, ppl_traces = compute_burstiness_trace(full_tokens, config)
    
    burst_valid = len(positions) >= config.MIN_TARGETS_PER_DOC
    
    if burst_valid:
        burst_metrics = compute_burstiness_metrics(I_L_trace, config)
        
        # Store trace
        for i, (pos, I_L) in enumerate(zip(positions, I_L_trace)):
            burst_traces.append({
                'doc_id': row['doc_id'],
                'age_months': row['age_months'],
                'age_group': row['age_group'],
                'target_idx': i,
                'target_pos': pos,
                'target_pos_frac': pos / n_tokens,
                'I_L': I_L,
            })
    else:
        burst_metrics = {
            'spike_mass': np.nan,
            'spike_count': np.nan,
            'cv': np.nan,
            'kurtosis': np.nan,
            'I_L_mean': np.nan,
            'I_L_max': np.nan,
            'I_L_std': np.nan,
            'n_targets': 0,
        }
    
    # Combine metrics
    doc_metrics.append({
        'doc_id': row['doc_id'],
        'age_months': row['age_months'],
        'age_group': row['age_group'],
        'word_count': row['word_count'],
        'n_tokens': n_tokens,
        
        # End-of-doc metrics (v3)
        'ppl_min_ctx': metrics_fixed['ppl_at_min_ctx'],
        'ppl_32': metrics_fixed['ppl_at_32'],
        'ppl_max_fixed': metrics_fixed['ppl_at_max_ctx'],
        'total_benefit_fixed': metrics_fixed['total_benefit'],
        'benefit_ok_fixed': metrics_fixed['benefit_ok'],
        'half_life_fixed': metrics_fixed['half_life'],
        'early_drop_pct_fixed': metrics_fixed['early_drop_pct'],
        'early_slope': metrics_fixed['early_slope'],
        
        # Burstiness metrics (NEW)
        'burst_valid': burst_valid,
        'burst_n_targets': burst_metrics['n_targets'],
        'burst_I_L_mean': burst_metrics['I_L_mean'],
        'burst_I_L_max': burst_metrics['I_L_max'],
        'burst_I_L_std': burst_metrics['I_L_std'],
        'burst_spike_mass': burst_metrics['spike_mass'],
        'burst_spike_count': burst_metrics['spike_count'],
        'burst_cv': burst_metrics['cv'],
        'burst_kurtosis': burst_metrics['kurtosis'],
        'burst_target_positions': json.dumps(positions) if burst_valid else '[]',
    })

results_df = pd.DataFrame(all_results_endofdoc)
metrics_df = pd.DataFrame(doc_metrics)
traces_df = pd.DataFrame(burst_traces)

print(f"\nProcessed {len(metrics_df)} documents")
print(f"Documents with valid burstiness analysis: {metrics_df['burst_valid'].sum()}")
print(f"Total target observations: {len(traces_df)}")

In [ ]:
# ============================================================
# BURSTINESS RESULTS BY AGE GROUP
# ============================================================

def bootstrap_ci(values, n_bootstrap=1000, ci=0.95, statistic=np.mean):
    """Compute bootstrap CI for a statistic."""
    values = np.array(values)
    values = values[~np.isnan(values)]
    
    if len(values) < 3:
        return np.nan, np.nan, np.nan
    
    rng = np.random.default_rng(42)
    boot_stats = []
    
    for _ in range(n_bootstrap):
        sample = rng.choice(values, size=len(values), replace=True)
        boot_stats.append(statistic(sample))
    
    alpha = (1 - ci) / 2
    ci_low = np.percentile(boot_stats, alpha * 100)
    ci_high = np.percentile(boot_stats, (1 - alpha) * 100)
    
    return statistic(values), ci_low, ci_high


print("=" * 70)
print("BURSTINESS RESULTS BY AGE GROUP")
print("=" * 70)

burst_valid_df = metrics_df[metrics_df['burst_valid']].copy()
print(f"\nDocuments with valid burstiness data: {len(burst_valid_df)}")

burst_summary = []

for ag in age_groups:
    ag_df = burst_valid_df[burst_valid_df['age_group'] == ag]
    
    if len(ag_df) < 5:
        continue
    
    result = {'age_group': ag, 'n': len(ag_df)}
    
    for metric in ['burst_I_L_mean', 'burst_spike_mass', 'burst_cv', 'burst_kurtosis']:
        mean, ci_low, ci_high = bootstrap_ci(
            ag_df[metric].values,
            n_bootstrap=config.N_BOOTSTRAP,
            ci=config.CI_LEVEL
        )
        result[f'{metric}_mean'] = mean
        result[f'{metric}_ci_low'] = ci_low
        result[f'{metric}_ci_high'] = ci_high
    
    burst_summary.append(result)

burst_summary_df = pd.DataFrame(burst_summary)

print("\n--- Mean Long-Range Influence (I_512) ---")
for _, row in burst_summary_df.iterrows():
    print(f"{row['age_group']}: {row['burst_I_L_mean_mean']:.2f} "
          f"[{row['burst_I_L_mean_ci_low']:.2f}, {row['burst_I_L_mean_ci_high']:.2f}] "
          f"(n={row['n']})")

print("\n--- Spike Mass (concentration in top 10%) ---")
for _, row in burst_summary_df.iterrows():
    print(f"{row['age_group']}: {row['burst_spike_mass_mean']:.3f} "
          f"[{row['burst_spike_mass_ci_low']:.3f}, {row['burst_spike_mass_ci_high']:.3f}]")

print("\n--- Coefficient of Variation (higher = burstier) ---")
for _, row in burst_summary_df.iterrows():
    print(f"{row['age_group']}: {row['burst_cv_mean']:.3f} "
          f"[{row['burst_cv_ci_low']:.3f}, {row['burst_cv_ci_high']:.3f}]")

print("\n--- Kurtosis (higher = more spiky tails) ---")
for _, row in burst_summary_df.iterrows():
    if not np.isnan(row['burst_kurtosis_mean']):
        print(f"{row['age_group']}: {row['burst_kurtosis_mean']:.3f} "
              f"[{row['burst_kurtosis_ci_low']:.3f}, {row['burst_kurtosis_ci_high']:.3f}]")

In [ ]:
# ============================================================
# STATISTICAL TESTS FOR BURSTINESS
# ============================================================

print("\n" + "=" * 70)
print("BURSTINESS STATISTICAL TESTS")
print("=" * 70)

print("\n--- Correlations with Age (Spearman) ---")
for metric in ['burst_I_L_mean', 'burst_spike_mass', 'burst_cv', 'burst_kurtosis']:
    valid = burst_valid_df[[metric, 'age_months']].dropna()
    if len(valid) > 10:
        rho, p = stats.spearmanr(valid['age_months'], valid[metric])
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"{metric:<25}: rho={rho:+.3f}, p={p:.4f} {sig}")

print("\n--- Kruskal-Wallis (overall age group effect) ---")
for metric in ['burst_I_L_mean', 'burst_spike_mass', 'burst_cv']:
    groups = [burst_valid_df[burst_valid_df['age_group'] == ag][metric].dropna().values 
              for ag in age_groups if len(burst_valid_df[burst_valid_df['age_group'] == ag]) > 0]
    groups = [g for g in groups if len(g) >= 3]
    
    if len(groups) >= 2:
        h, p = stats.kruskal(*groups)
        sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
        print(f"{metric:<25}: H={h:.2f}, p={p:.4f} {sig}")

In [ ]:
# ============================================================
# EXAMPLE TRACES: YOUNG vs OLDER DOCUMENT
# ============================================================

print("\n" + "=" * 70)
print("EXAMPLE DOCUMENT TRACES")
print("=" * 70)

# Find example docs with good trace data
young_docs = burst_valid_df[burst_valid_df['age_group'] == '4-6yr'].nlargest(5, 'burst_n_targets')
older_docs = burst_valid_df[burst_valid_df['age_group'].isin(['8-10yr', '10+yr'])].nlargest(5, 'burst_n_targets')

if len(young_docs) > 0 and len(older_docs) > 0:
    young_example = young_docs.iloc[0]
    older_example = older_docs.iloc[0]
    
    print(f"\nYoung example: {young_example['doc_id']} (age={young_example['age_months']}mo, "
          f"n_targets={young_example['burst_n_targets']})")
    print(f"  I_L mean={young_example['burst_I_L_mean']:.2f}, "
          f"spike_mass={young_example['burst_spike_mass']:.3f}, "
          f"CV={young_example['burst_cv']:.3f}")
    
    print(f"\nOlder example: {older_example['doc_id']} (age={older_example['age_months']}mo, "
          f"n_targets={older_example['burst_n_targets']})")
    print(f"  I_L mean={older_example['burst_I_L_mean']:.2f}, "
          f"spike_mass={older_example['burst_spike_mass']:.3f}, "
          f"CV={older_example['burst_cv']:.3f}")
    
    # Get traces for these docs
    young_trace = traces_df[traces_df['doc_id'] == young_example['doc_id']]
    older_trace = traces_df[traces_df['doc_id'] == older_example['doc_id']]
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    ax = axes[0]
    ax.plot(young_trace['target_pos_frac'], young_trace['I_L'], 'o-', 
            color='#e74c3c', linewidth=2, markersize=8, label=f"Young ({young_example['age_months']}mo)")
    ax.axhline(young_example['burst_I_L_mean'], color='#e74c3c', linestyle='--', alpha=0.5)
    ax.set_xlabel('Position in Document (fraction)')
    ax.set_ylabel('I_512 (long-range influence)')
    ax.set_title(f"Young Child: {young_example['doc_id']}\n"
                 f"spike_mass={young_example['burst_spike_mass']:.3f}, CV={young_example['burst_cv']:.3f}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    ax = axes[1]
    ax.plot(older_trace['target_pos_frac'], older_trace['I_L'], 'o-',
            color='#27ae60', linewidth=2, markersize=8, label=f"Older ({older_example['age_months']}mo)")
    ax.axhline(older_example['burst_I_L_mean'], color='#27ae60', linestyle='--', alpha=0.5)
    ax.set_xlabel('Position in Document (fraction)')
    ax.set_ylabel('I_512 (long-range influence)')
    ax.set_title(f"Older Child: {older_example['doc_id']}\n"
                 f"spike_mass={older_example['burst_spike_mass']:.3f}, CV={older_example['burst_cv']:.3f}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('ecsc_burst_example_traces.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Not enough examples for comparison plot.")

In [ ]:
# ============================================================
# BURSTINESS VISUALIZATION
# ============================================================

colors = {'4-6yr': '#e74c3c', '6-8yr': '#f39c12', '8-10yr': '#27ae60', '10+yr': '#3498db'}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. I_L mean by age group
ax = axes[0, 0]
x_pos = range(len(burst_summary_df))
means = burst_summary_df['burst_I_L_mean_mean'].values
ci_low = burst_summary_df['burst_I_L_mean_ci_low'].values
ci_high = burst_summary_df['burst_I_L_mean_ci_high'].values
errors = [means - ci_low, ci_high - means]

bars = ax.bar(x_pos, means, yerr=errors, capsize=8,
              color=[colors.get(ag, 'gray') for ag in burst_summary_df['age_group']],
              alpha=0.7, edgecolor='black')
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{row['age_group']}\n(n={row['n']})" for _, row in burst_summary_df.iterrows()])
ax.set_ylabel('Mean I_512')
ax.set_title('Mean Long-Range Influence\n(higher = more benefit from distant context)')
ax.grid(True, alpha=0.3, axis='y')

# 2. Spike mass by age group
ax = axes[0, 1]
means = burst_summary_df['burst_spike_mass_mean'].values
ci_low = burst_summary_df['burst_spike_mass_ci_low'].values
ci_high = burst_summary_df['burst_spike_mass_ci_high'].values
errors = [means - ci_low, ci_high - means]

bars = ax.bar(x_pos, means, yerr=errors, capsize=8,
              color=[colors.get(ag, 'gray') for ag in burst_summary_df['age_group']],
              alpha=0.7, edgecolor='black')
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{row['age_group']}" for _, row in burst_summary_df.iterrows()])
ax.set_ylabel('Spike Mass')
ax.set_title('Spike Mass (top 10% concentration)\n(higher = more concentrated influence)')
ax.grid(True, alpha=0.3, axis='y')

# 3. CV by age group
ax = axes[0, 2]
means = burst_summary_df['burst_cv_mean'].values
ci_low = burst_summary_df['burst_cv_ci_low'].values
ci_high = burst_summary_df['burst_cv_ci_high'].values
errors = [means - ci_low, ci_high - means]

bars = ax.bar(x_pos, means, yerr=errors, capsize=8,
              color=[colors.get(ag, 'gray') for ag in burst_summary_df['age_group']],
              alpha=0.7, edgecolor='black')
ax.set_xticks(x_pos)
ax.set_xticklabels([f"{row['age_group']}" for _, row in burst_summary_df.iterrows()])
ax.set_ylabel('CV (std/mean)')
ax.set_title('Coefficient of Variation\n(higher = burstier pattern)')
ax.grid(True, alpha=0.3, axis='y')

# 4. Scatter: age vs I_L mean
ax = axes[1, 0]
for ag in age_groups:
    ag_data = burst_valid_df[burst_valid_df['age_group'] == ag]
    ax.scatter(ag_data['age_months'], ag_data['burst_I_L_mean'],
               alpha=0.5, label=ag, color=colors.get(ag, 'gray'), s=30)

# Trend line
valid = burst_valid_df[['age_months', 'burst_I_L_mean']].dropna()
if len(valid) > 10:
    z = np.polyfit(valid['age_months'], valid['burst_I_L_mean'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(valid['age_months'].min(), valid['age_months'].max(), 100)
    ax.plot(x_line, p(x_line), 'k--', alpha=0.7, linewidth=2)

ax.set_xlabel('Age (months)')
ax.set_ylabel('Mean I_512')
ax.set_title('Age vs Mean Long-Range Influence')
ax.legend()
ax.grid(True, alpha=0.3)

# 5. Scatter: age vs spike mass
ax = axes[1, 1]
for ag in age_groups:
    ag_data = burst_valid_df[burst_valid_df['age_group'] == ag]
    ax.scatter(ag_data['age_months'], ag_data['burst_spike_mass'],
               alpha=0.5, label=ag, color=colors.get(ag, 'gray'), s=30)

ax.set_xlabel('Age (months)')
ax.set_ylabel('Spike Mass')
ax.set_title('Age vs Spike Mass')
ax.legend()
ax.grid(True, alpha=0.3)

# 6. Box plot of I_L distributions
ax = axes[1, 2]
box_data = [burst_valid_df[burst_valid_df['age_group'] == ag]['burst_I_L_mean'].dropna().values 
            for ag in age_groups if ag in burst_valid_df['age_group'].values]
box_labels = [ag for ag in age_groups if ag in burst_valid_df['age_group'].values]

bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True)
for patch, ag in zip(bp['boxes'], box_labels):
    patch.set_facecolor(colors.get(ag, 'gray'))
    patch.set_alpha(0.7)

ax.set_ylabel('Mean I_512')
ax.set_title('Distribution of Long-Range Influence\nby Age Group')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('ecsc_burstiness_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# END-OF-DOC RESULTS (v3 compatibility)
# ============================================================

print("\n" + "=" * 70)
print("END-OF-DOC RESULTS (v3 metrics)")
print("=" * 70)

endofdoc_summary = []

for ag in age_groups:
    ag_df = metrics_df[metrics_df['age_group'] == ag]
    
    if len(ag_df) < 5:
        continue
    
    result = {'age_group': ag, 'n': len(ag_df)}
    
    for metric in ['half_life_fixed', 'early_slope', 'early_drop_pct_fixed', 'ppl_min_ctx']:
        mean, ci_low, ci_high = bootstrap_ci(
            ag_df[metric].values,
            n_bootstrap=config.N_BOOTSTRAP,
            ci=config.CI_LEVEL
        )
        result[f'{metric}_mean'] = mean
        result[f'{metric}_ci_low'] = ci_low
        result[f'{metric}_ci_high'] = ci_high
    
    endofdoc_summary.append(result)

endofdoc_summary_df = pd.DataFrame(endofdoc_summary)

print("\n--- Half-Life (end-of-doc, fixed max=512) ---")
for _, row in endofdoc_summary_df.iterrows():
    print(f"{row['age_group']}: {row['half_life_fixed_mean']:.1f} "
          f"[{row['half_life_fixed_ci_low']:.1f}, {row['half_life_fixed_ci_high']:.1f}] "
          f"(n={row['n']})")

print("\n--- Early Drop % ---")
for _, row in endofdoc_summary_df.iterrows():
    print(f"{row['age_group']}: {row['early_drop_pct_fixed_mean']:.1f}% "
          f"[{row['early_drop_pct_fixed_ci_low']:.1f}, {row['early_drop_pct_fixed_ci_high']:.1f}]")

In [ ]:
# ============================================================
# SAVE RESULTS
# ============================================================

# Per-document metrics (combined)
metrics_df.to_csv('ecsc_age_v4_metrics.csv', index=False)

# Burstiness traces (per target)
traces_df.to_csv('ecsc_age_v4_traces.csv', index=False)

# Burstiness summary by age group
burst_summary_df.to_csv('ecsc_age_v4_burst_summary.csv', index=False)

# End-of-doc summary by age group
endofdoc_summary_df.to_csv('ecsc_age_v4_endofdoc_summary.csv', index=False)

# Config
config_dict = {
    'MIN_WORDS': config.MIN_WORDS,
    'K_TARGETS': config.K_TARGETS,
    'TARGET_START_FRAC': config.TARGET_START_FRAC,
    'TARGET_END_FRAC': config.TARGET_END_FRAC,
    'TARGET_REGION_TOKENS': config.TARGET_REGION_TOKENS,
    'CONTEXT_GRID_BURST': config.CONTEXT_GRID_BURST,
    'SHORT_CONTEXT': config.SHORT_CONTEXT,
    'LONG_CONTEXT': config.LONG_CONTEXT,
    'SPIKE_PERCENTILE': config.SPIKE_PERCENTILE,
    'SPIKE_MASS_TOP_FRAC': config.SPIKE_MASS_TOP_FRAC,
    'MIN_TARGETS_PER_DOC': config.MIN_TARGETS_PER_DOC,
    'MAX_CONTEXT_FIXED': config.MAX_CONTEXT_FIXED,
    'N_BOOTSTRAP': config.N_BOOTSTRAP,
}
with open('ecsc_age_v4_config.json', 'w') as f:
    json.dump(config_dict, f, indent=2)

print("Saved:")
print("  - ecsc_age_v4_metrics.csv (per-document)")
print("  - ecsc_age_v4_traces.csv (per-target I_L values)")
print("  - ecsc_age_v4_burst_summary.csv (burstiness by age group)")
print("  - ecsc_age_v4_endofdoc_summary.csv (end-of-doc by age group)")
print("  - ecsc_age_v4_config.json")
print("  - ecsc_burstiness_summary.png")
print("  - ecsc_burst_example_traces.png")

In [ ]:
from google.colab import files
import os

files.download('ecsc_age_v4_metrics.csv')
files.download('ecsc_age_v4_traces.csv')
files.download('ecsc_age_v4_burst_summary.csv')
files.download('ecsc_age_v4_endofdoc_summary.csv')
files.download('ecsc_age_v4_config.json')
files.download('ecsc_burstiness_summary.png')

if os.path.exists('ecsc_burst_example_traces.png'):
    files.download('ecsc_burst_example_traces.png')

## Summary

### New Burstiness Analysis

**Method:**
- K=10 target positions per document (15%-90% of narrative)
- At each target: compute I_512 = ppl(32) - ppl(512)
- I_512 measures additional benefit from long context beyond local coherence

**Burstiness Metrics:**
- `spike_mass`: fraction of total I_L in top 10% of targets
- `spike_count`: number of targets above 95th percentile
- `CV`: coefficient of variation (std/mean)
- `kurtosis`: excess kurtosis (spiky tails)

**Interpretation:**
- High spike_mass / CV / kurtosis → event-like retrieval (concentrated spikes)
- Low spike_mass / CV → diffuse contextual influence (spread smoothly)

### Outputs

| File | Description |
|------|-------------|
| `ecsc_age_v4_metrics.csv` | Per-document metrics (end-of-doc + burstiness) |
| `ecsc_age_v4_traces.csv` | Per-target I_L values (for detailed analysis) |
| `ecsc_age_v4_burst_summary.csv` | Burstiness metrics by age group with CIs |
| `ecsc_age_v4_endofdoc_summary.csv` | End-of-doc metrics by age group (v3 compat) |
| `ecsc_burstiness_summary.png` | Main burstiness visualization |
| `ecsc_burst_example_traces.png` | Example young vs older document traces |